# AdventureWorks products on Databricks, with the tabular model

The same catalog as [`adventureworks.ipynb`](adventureworks.ipynb) and a
different question. That notebook joins four tables and asks what a customer
will do next; this one reads a single table and fills in a column that is
missing on some of its rows.

`kumo-tabular` scores rows of one table. It reads no relationships and no time
windows, so it needs neither the relational engine nor a graph: labelled rows
go in one side, the rows you want answered go in the other.

You need the AdventureWorks products table (`adventureworks_products`) in a
Unity Catalog schema, and a `kumo-tabular` Model Serving endpoint.

In [ ]:
dbutils.widgets.text(
    'install',
    'structured-data-models-client[databricks,databricks-serving]',
    'pip install',
)
dbutils.widgets.text('catalog', 'main', 'Catalog')
dbutils.widgets.text('schema', 'adventureworks', 'Schema')
dbutils.widgets.text('table', 'adventureworks_products', 'Products table')
dbutils.widgets.text('endpoint', '', 'Model Serving endpoint')
dbutils.widgets.text('warehouse_id', '', 'SQL warehouse ID, or empty for Spark')

install = dbutils.widgets.get('install')

# The runtime's packages are built against the NumPy it ships; pinning it keeps
# the install from upgrading NumPy beneath them, which stops Python starting.
import numpy

keep_numpy = f'numpy=={numpy.__version__}'

In [ ]:
%pip install --only-binary=:all: $install $keep_numpy

In [ ]:
dbutils.library.restartPython()

In [ ]:
import json
import os
import platform
import sys
from importlib.metadata import PackageNotFoundError, version


def installed(package):
    try:
        return version(package)
    except PackageNotFoundError:
        return None


ENVIRONMENT = {
    'runtime': os.environ.get('DATABRICKS_RUNTIME_VERSION'),
    'python': sys.version.split()[0],
    'machine': platform.machine(),
    **{
        package: installed(package)
        for package in (
            'structured-data-models-client',
            'structured-data-models-connectors',
            'numpy',
            'pandas',
            'pyarrow',
            'pydantic',
            'databricks-sdk',
            'databricks-sql-connector',
        )
    },
}
print(json.dumps(ENVIRONMENT, indent=1))

## 1. Read the one table

`read_table` returns a DataFrame from a connection, for a single flat table,
without building a source or a schema.

With no warehouse named this is the notebook's own `spark`; with one named,
`warehouse()` opens a connection to it as the notebook's `WorkspaceClient`.

This reads the table, which is the difference from the relational notebook:
there the client samples a graph with SQL and only the rows a prediction needs
leave the catalog. One table small enough to score is simply read.

In [ ]:
from databricks.sdk import WorkspaceClient
from sdm_connectors import read_table
from sdm_connectors.databricks import warehouse

catalog = dbutils.widgets.get('catalog')
schema_name = dbutils.widgets.get('schema')
table_name = dbutils.widgets.get('table')
endpoint = dbutils.widgets.get('endpoint')
warehouse_id = dbutils.widgets.get('warehouse_id')
if not endpoint:
    raise ValueError(
        'Set the endpoint widget to a Kumo Tabular serving endpoint.'
    )

workspace = WorkspaceClient()
if warehouse_id:
    connection = warehouse(warehouse_id, workspace=workspace)
else:
    connection = spark

products = read_table(
    connection, table=f'{catalog}.{schema_name}.{table_name}'
)
print(f'{len(products)} rows, {len(products.columns)} columns')
products.head()

## 2. Say which rows carry the answer

The model reads one table, so the split is the whole input: rows where the
column is known teach it, and rows where it is missing are the question.

`rowguid` is a per-row UUID AdventureWorks carries for replication. It
identifies a row and says nothing about it, so it goes; so does the name, which
names the row rather than describing it. A column left out is neither read nor
sent.

In [ ]:
TARGET = 'Color'
IGNORED = ['rowguid', 'Name', 'ProductNumber', 'ModifiedDate']

columns = [c for c in products.columns if c not in IGNORED]
known = products.loc[products[TARGET].notna(), columns]
unknown = products.loc[products[TARGET].isna(), columns].drop(columns=[TARGET])

print(f'{len(known)} rows carry a {TARGET}, {len(unknown)} do not')
print(known[TARGET].value_counts().to_string())

## 3. Ask

`Request` is the whole tabular interface: the labelled rows, the rows to score,
which column is the answer, and what kind of question it is. No schema, no
query language, no graph.

`task='classification'` is stated rather than inferred. The client would guess
from what the column holds, and says so when it does; stating it settles the
case where whole numbers could be labels or quantities.

In [ ]:
from sdm_client import KumoTabular, Request, SDMClient

request = Request(
    context=known,
    predict=unknown,
    target=TARGET,
    task='classification',
)

with SDMClient.for_databricks_serving(
    endpoint, workspace_client=workspace
) as client:
    answer = client.predict(request, model=KumoTabular())

answer.to_pandas().head(10)

One row per row you asked about, keyed by `row_index`, carrying the predicted
class and a probability column per class.

The probabilities are the model's, not a calibrated score: read them as an
ordering over the classes rather than as a frequency.

## 4. The same table, a number instead of a label

Changing the question means changing the target and the task. Everything else
is the same call, which is the point of the tabular model.

Here the answer is a price, so the task is a regression and the result carries
one `prediction` column and no probabilities.

In [ ]:
priced = products.loc[products['ListPrice'] > 0, columns]
to_price = priced.tail(20).drop(columns=['ListPrice'])

regression = Request(
    context=priced.head(len(priced) - 20),
    predict=to_price,
    target='ListPrice',
    task='regression',
)

with SDMClient.for_databricks_serving(
    endpoint, workspace_client=workspace
) as client:
    print(client.predict(regression, model=KumoTabular()).to_pandas().head())

## Which model to reach for

Use `kumo-tabular` when the answer is a column of the table you already have,
and the rows are independent of each other.

Use `kumo-relational` when the answer depends on what an entity is linked to,
or on when things happened: a forecast, a churn window, what someone buys next.
That is [`adventureworks.ipynb`](adventureworks.ipynb), over these same
tables.